In [26]:
import sqlite3
import pandas as pd

In [27]:
# Match the genres of the movies with the titles of the movies in ascending order according to genre
# Output: 11805 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT res.genre AS genre, m.title AS movie_title
FROM (
    SELECT g.name AS genre, hg.movie_id AS movie_id 
    FROM genre g LEFT OUTER JOIN hasGenre hg ON g.id = hg.genre_id
    GROUP BY g.name, hg.movie_id
) AS res, movie m 
WHERE m.id = res.movie_id
GROUP BY res.genre, m.title
ORDER BY res.genre;
"""

df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,genre,movie_title
0,Action,...All the Marbles
1,Action,...tick... tick... tick...
2,Action,1492: Conquest of Paradise
3,Action,15 Minutes
4,Action,1941
...,...,...
11800,Music,You Were Never Lovelier
11801,Music,You'll Never Get Rich
11802,Music,Young at Heart
11803,Music,Ziggy Stardust and the Spiders from Mars


In [28]:
# Show me the first 10 lines from the movie genre association with the titles in descending order according to the movie title
# Output: 10 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT * 
FROM (SELECT res.genre AS genre, m.title AS movie_title
	FROM (SELECT g.name AS genre, hg.movie_id AS movie_id 
		FROM genre g LEFT OUTER JOIN hasGenre hg ON g.id = hg.genre_id
		GROUP BY g.name, hg.movie_id) AS res, movie m 
	WHERE m.id = res.movie_id
	GROUP BY res.genre, m.title) AS res
	ORDER BY res.movie_title DESC
    LIMIT 10;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,genre,movie_title
0,Drama,�elary
1,Drama,�Round Midnight
2,Action,xXx
3,Adventure,xXx
4,Drama,ivans xtc.
5,Action,eXistenZ
6,Action,Zulu
7,Drama,Zulu
8,Action,Zu: Warriors from the Magic Mountain
9,Drama,Zu: Warriors from the Magic Mountain


In [29]:
# Find out which movies belong to the "Animation" category
# Output: 262 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title
FROM (SELECT g.name AS genre, hg.movie_id AS movie_id 
	FROM genre g LEFT OUTER JOIN hasGenre hg ON g.id = hg.genre_id
	GROUP BY g.name, hg.movie_id) AS res, movie m 
WHERE m.id = res.movie_id
GROUP BY res.genre, m.title
HAVING res.genre LIKE '%ANIMATION%';
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title
0,A Bug's Life
1,A Charlie Brown Christmas
2,A Close Shave
3,A Goofy Movie
4,A Grand Day Out
...,...
257,Winnie the Pooh and the Blustery Day
258,Wizards
259,Wonderful Days
260,Yellow Submarine


In [30]:
# Find out which movies each production company has released
# Output: 18544 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT pc.production_company_name AS production_company_name, m.title AS movie_title 
FROM(SELECT pc.name AS production_company_name, hpc.movie_id 
	FROM productioncompany pc 
	INNER JOIN hasProductioncompany hpc ON pc.id = hpc.pc_id
	GROUP BY pc.name, hpc.movie_id) AS pc, movie m 
WHERE m.id = pc.movie_id
GROUP BY pc.production_company_name, m.title
ORDER BY pc.production_company_name DESC;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,production_company_name,movie_title
0,Česká televize,Dark Blue World
1,Új Budapest Filmstudió,Magic Hunter
2,Österreichischer Rundfunk (ORF),Colonel Redl
3,Österreichischer Rundfunk (ORF),Downfall
4,Österreichischer Rundfunk (ORF),Sunshine
...,...,...
18539,12 Gauge Productions,Dead Man
18540,101st Street Films,Chairman of the Board
18541,.406 Production,Schizopolis
18542,", Flaminia Produzioni Cinematografiche",The Living Dead at Manchester Morgue


In [31]:
# Find me the total budget for the movies produced by Pixar.
# Output: 1 row 

conn = sqlite3.connect('movies_database.db')

query = """
SELECT SUM(budget.budget) AS Pixar_overall_budget
FROM(SELECT SUM(budget) AS budget
	FROM(SELECT pc.name AS production_company_name, hpc.movie_id 
		FROM productioncompany pc INNER JOIN hasProductioncompany hpc ON pc.id = hpc.pc_id
		GROUP BY pc.name, hpc.movie_id) AS pc, movie m 
	WHERE m.id = pc.movie_id
	GROUP BY pc.production_company_name, m.title
	HAVING pc.production_company_name LIKE '%PIXAR%') AS budget

"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,Pixar_overall_budget
0,541000000


In [32]:
# Find me the total revenue of Pixar from the movies it produced.
# Output: 1 row

conn = sqlite3.connect('movies_database.db')

query = """
SELECT SUM(CAST(result.revenue AS bigint)) AS Pixar_overall_revenue
FROM(SELECT SUM(revenue) AS revenue
	FROM(SELECT pc.name AS production_company_name, hpc.movie_id 
		FROM productioncompany pc INNER JOIN hasProductioncompany hpc ON pc.id = hpc.pc_id
		GROUP BY pc.name, hpc.movie_id) AS pc, movie m 
	WHERE m.id = pc.movie_id
	GROUP BY pc.production_company_name, m.title
	HAVING pc.production_company_name LIKE '%PIXAR%') AS result
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,Pixar_overall_revenue
0,3368773645


In [33]:
# Find out which movies the production company "Pixar" has released.
# And show them to me in descending order based on the name of the movie.
# Output: 6 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_produced_by_Pixar
FROM(SELECT pc.name AS production_company_name, hpc.movie_id 
    FROM productioncompany pc INNER JOIN hasProductioncompany hpc ON pc.id = hpc.pc_id
    GROUP BY pc.name, hpc.movie_id) AS pc, movie m 
WHERE m.id = pc.movie_id
GROUP BY pc.production_company_name, m.title
HAVING pc.production_company_name LIKE '%PIXAR%'
ORDER BY m.title DESC;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_produced_by_Pixar
0,Toy Story 2
1,Toy Story
2,The Incredibles
3,"Monsters, Inc."
4,Finding Nemo
5,A Bug's Life


In [34]:
# Find me the average budget for the movies produced by Warner Bros
# Output: 1 row 
conn = sqlite3.connect('movies_database.db')

query = """
SELECT AVG(CAST(res.budget AS bigint)) AS "Warner Bross average budget"
FROM(SELECT AVG(budget) AS budget
	FROM(SELECT pc.name AS production_company_name, hpc.movie_id 
		FROM productioncompany pc INNER JOIN hasProductioncompany hpc ON pc.id = hpc.pc_id
		GROUP BY pc.name, hpc.movie_id) AS pc, movie m 
	WHERE m.id = pc.movie_id
	GROUP BY pc.production_company_name, m.title
	HAVING pc.production_company_name LIKE '%WARNER BROS%') AS res
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,Warner Bross average budget
0,1.881285e+07


In [35]:
# In which movies does Bob Dylan act?
# Output: 5 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT movie.title AS movie_with_Bob_Dylan
FROM movie INNER JOIN movie_cast
    ON movie_cast.name = 'Bob Dylan' AND movie_cast.movie_id = movie.id;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_with_Bob_Dylan
0,Dont Look Back
1,Pat Garrett & Billy the Kid
2,Masked and Anonymous
3,The Last Waltz
4,Message to Love: The Isle of Wight Festival


In [36]:
# Which actors star in the movie "The Name of the Rose"?
# Output: 37 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT mc.name AS "Casting of 'The name of the Rose'"
FROM movie m INNER JOIN movie_cast mc 
ON m.id = mc.movie_id AND m.title LIKE '%The name of the Rose%'
ORDER BY mc.name;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,Casting of 'The name of the Rose'
0,Andrew Birkin
1,Aristide Caporale
2,Armando Marra
3,Christian Slater
4,Donald O'Brien
5,Elya Baskin
6,F. Murray Abraham
7,Fabio Carfora
8,Feodor Chaliapin Jr.
9,Francesco Scali


In [37]:
# Who worked on the film "Prince of Darkness"?
# Output: 17 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT mc.job AS job, mc.name AS name
FROM movie_crew mc 
INNER JOIN movie m 
ON mc.movie_id = m.id AND m.title = 'Prince of Darkness';
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,job,name
0,Director,John Carpenter
1,Screenplay,John Carpenter
2,Original Music Composer,John Carpenter
3,Original Music Composer,Alan Howarth
4,Producer,Larry J. Franco
5,Director of Photography,Gary B. Kibbe
6,Executive Producer,Andre Blay
7,Executive Producer,Shep Gordon
8,Editor,Steve Mirkovich
9,Casting,Linda Francis


In [38]:
# Movies from 2013 and onward
# Output: 2 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title, m.release_date AS release_date
FROM movie m
WHERE m.release_date >= '2013';
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,release_date
0,Camille Claudel 1915,2013-03-13
1,The Sleepover,2013-10-12


In [39]:
# Movies in the original language in French, sorted by the title of the movie.
# Output: 441 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title
FROM movie m
WHERE m.original_language LIKE '%FR%'
ORDER BY m.title;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title
0,...And God Created Woman
1,2 or 3 Things I Know About Her
2,8 Women
3,A Chef in Love
4,A Heart in Winter
...,...
436,Z
437,Zazie dans le m�tro
438,Zero for Conduct
439,Zombie Lake


In [40]:
# Which characters has Harrison Ford portrayed?
# Output: 30 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT DISTINCT mc.character AS "Harrison Ford's characters"
FROM movie_cast mc INNER JOIN movie m 
ON mc.name LIKE '%Harrison Ford%' AND mc.movie_id = m.id 
WHERE mc.character IS NOT NULL
ORDER by mc.character;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,Harrison Ford's characters
0,Alexei Vostrikov
1,Allie Fox
2,Armless Soldier
3,Arrested Student (uncredited)
4,Bob Falfa
5,Colonel Lucas
6,Det. Capt. John Book
7,Dr. Kimble
8,Dr. Norman Spencer
9,Dr. Richard Walker


In [41]:
# What keywords does each movie contain?
# Output: 56780 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT DISTINCT m.title AS movie_title, result.name AS keyword_name
FROM (SELECT hk.movie_id AS movie_id, k.name 
	FROM haskeyword hk
	INNER JOIN keyword k 
	ON hk.keyword_id = k.id
	GROUP BY hk.movie_id, k.name) AS result
INNER JOIN movie m
ON result.movie_id = m.id
ORDER BY m.title DESC;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,keyword_name
0,�Three Amigos!,actor
1,�Three Amigos!,confusion
2,�Three Amigos!,hero
3,�Three Amigos!,mexico
4,�Three Amigos!,spoof
...,...,...
56775,"'night, Mother",epilepsy
56776,"'night, Mother",mother daughter relationship
56777,"'night, Mother",pulitzer prize
56778,"'night, Mother",suicide


In [42]:
# What keywords do the movies that contain the word 'TOY' in their title include?
# Output: 44 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title, result.name AS keyword_name
FROM (SELECT hk.movie_id AS movie_id, k.name 
	FROM haskeyword hk
	INNER JOIN keyword k 
	ON hk.keyword_id = k.id
	GROUP BY hk.movie_id, k.name) AS result
INNER JOIN movie m
ON result.movie_id = m.id
WHERE m.title LIKE '%TOY%'
ORDER BY m.title;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,keyword_name
0,Babes in Toyland,holiday
1,Babes in Toyland,toys
2,Babes in Toyland,kids and family
3,Giants and Toys,japanese new wave
4,The Toy,department store
5,The Toy,job termination
6,The Toy,rich kid
7,The Toy,richard pryor
8,The Toy,trophy wife
9,Toy Soldiers,high school


In [43]:
# Movies with the highest average user rating
# Output: 37 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT res.title AS movie_title, MAX(res.avg_rating) AS maxAvg_rating
FROM (SELECT m.title, AVG(r.rating) AS avg_rating
    FROM movie m
    INNER JOIN ratings r
    ON m.id = r.movie_id
    GROUP BY m.id, m.title) AS res 
GROUP BY res.title, res.avg_rating
HAVING res.avg_rating IN (SELECT MAX(res2.avg_rating) 
    FROM (SELECT m.title, AVG(r.rating) AS avg_rating
    FROM movie m
    INNER JOIN ratings r
    ON m.id = r.movie_id
    GROUP BY m.id, m.title) AS res2);
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,maxAvg_rating
0,29th Street,5.0
1,A Countess from Hong Kong,5.0
2,A Streetcar Named Desire,5.0
3,Anchorman: The Legend of Ron Burgundy,5.0
4,Around the World in Eighty Days,5.0
5,Brigham City,5.0
6,Frank Herbert's Dune,5.0
7,Gentlemen Prefer Blondes,5.0
8,JFK,5.0
9,Knight Moves,5.0


In [44]:
# Movies with the lowest average user rating
# Output: 8 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT res.title AS movie_title, MIN(res.avg_rating) AS minAvg_rating
FROM (SELECT m.title, AVG(r.rating) AS avg_rating
    FROM movie m
    INNER JOIN ratings r
    ON m.id = r.movie_id
    GROUP BY m.id, m.title) AS res 
GROUP BY res.title, res.avg_rating
HAVING res.avg_rating IN (SELECT MIN(res2.avg_rating) 
    FROM (SELECT m.title, AVG(r.rating) AS avg_rating
    FROM movie m
    INNER JOIN ratings r
    ON m.id = r.movie_id
    GROUP BY m.id, m.title) AS res2);
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,minAvg_rating
0,Crimson Tide,0.5
1,"Dude, Where�s My Car?",0.5
2,El Dorado,0.5
3,Finder's Fee,0.5
4,Prick Up Your Ears,0.5
5,Stranded,0.5
6,Tough and Deadly,0.5
7,Vatel,0.5


In [45]:
# The movie with the highest popularity
# Output: 1 row

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title, m.popularity AS max_popularity
FROM movie m 
WHERE m.popularity IN (SELECT MAX(m.popularity) FROM movie m);
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,max_popularity
0,Pulp Fiction,140.950236


In [46]:
# The movie with the lowest popularity
# Output: 1 row
conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title, m.popularity AS min_popularity
FROM movie m 
WHERE m.popularity IN (SELECT MIN(m.popularity) FROM movie m);
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,min_popularity
0,Night of the Zombies,0.0


In [47]:
# Short films from 10 to 30 minutes.
# Output: 19 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT m.title AS movie_title, m.runtime AS runtime
FROM movie m 
WHERE m.runtime BETWEEN 10 AND 30
ORDER BY m.runtime;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,runtime
0,A Trip to the Moon,14.0
1,Un chien andalou,16.0
2,The Farmer's Wife,18.0
3,The Immigrant,20.0
4,A Grand Day Out,23.0
5,Harvie Krumpet,23.0
6,Will Vinton's Claymation Christmas Celebration,24.0
7,A Charlie Brown Christmas,25.0
8,Some Folks Call It a Sling Blade,25.0
9,Winnie the Pooh and the Blustery Day,25.0


In [48]:
# Websites (Not Null) for short films from 10 to 30 minutes
# Output: 4 rows

conn = sqlite3.connect('movies_database.db')

query = """
SELECT res.movie_title AS movie_title, m.homepage AS homepage
FROM (
    SELECT m.title AS movie_title, m.runtime AS runtime
    FROM movie m 
    WHERE m.runtime BETWEEN 10 AND 30
) AS res
JOIN movie m ON res.movie_title = m.title
-- Εδώ προσθέτουμε τα φίλτρα για να μην είναι κενό το homepage
WHERE m.homepage IS NOT NULL 
  AND m.homepage != ''
ORDER BY res.runtime;
"""
df_results = pd.read_sql_query(query, conn)
display(df_results)

conn.close()

,movie_title,homepage
0,The Farmer's Wife,http://www.thefarmerswifefilm.co.uk/
1,A Grand Day Out,http://www.wallaceandgromit.com/films/a-grand-...
2,The Wrong Trousers,http://www.wallaceandgromit.com/films/the-wron...
3,A Close Shave,http://www.wallaceandgromit.com/films/a-close-...
